# Building a RAG System with Gemini Embeddings, ChromaDB, and Groq

**Purpose:** learn the complete retrieval-to-generation path in one notebook, with explanations before the code that uses each component.

This notebook uses **Gemini for embeddings**, **ChromaDB for vector storage and retrieval**, **LangChain for common interfaces and chains**, and **Groq for answer generation**. These roles are kept separate throughout the notebook so that one model variable does not silently replace another.

## Learning outcomes

By the end, you should be able to explain and implement:

- The ingestion → chunking → embedding → indexing → retrieval → generation workflow.
- Why `Document` objects and metadata are useful.
- What an embedding adapter does and why query/document formatting matters.
- The difference between an embedding cache and a vector database.
- Persistent ChromaDB storage, cosine distance, similarity search, metadata filtering, and a retriever.
- One main LCEL RAG pipeline, plus an explanation of older high-level chain helpers.

> **Execution note:** run the notebook from top to bottom. API smoke tests and embedding generation make external API calls and may consume quota or incur cost. No cells in this cleaned copy have pre-existing outputs; those outputs were cleared because the original notebook had cells executed out of order.

## 1. Install dependencies

Use this in the project terminal, not as a Python cell, if the packages are not already installed. The commands below assume you use `uv` and an existing project.

```cmd
uv add google-genai langchain-core langchain-text-splitters langchain-chroma chromadb langchain-groq python-dotenv
```

If the project uses a different dependency manager, install the equivalent packages in the same Python environment used by Jupyter. Restart the notebook kernel after installing or upgrading packages.

**Why these packages?** `google-genai` calls Gemini; `langchain-core` provides `Document`, embedding interfaces, prompts, and runnables; `langchain-text-splitters` chunks text; `langchain-chroma` connects LangChain to ChromaDB; `chromadb` provides the database client; `langchain-groq` connects to Groq; `python-dotenv` loads local environment variables.

## 2. Configure credentials safely

Create a local `.env` file in the project directory. It should contain your own API keys; never paste actual secrets into a notebook cell or commit them to Git.

```env
GEMINI_API_KEY=replace_with_your_gemini_key
GROQ_API_KEY=replace_with_your_groq_key
```

Also put `.env` in `.gitignore`, along with generated local data such as `artifacts/` and the ChromaDB directory. The next cell loads the file and checks that both variables exist **without printing their values**.

## 3. Import the components used in this notebook

This cell imports only the components needed for the main workflow. Imports make Python names available; they do not make API calls or build the RAG pipeline.

The Chroma integration comes from the dedicated `langchain-chroma` package (`langchain_chroma`), rather than the older `langchain_community.vectorstores` import path.

In [2]:
import json
import os
from pathlib import Path

import chromadb
from dotenv import find_dotenv, load_dotenv
from google import genai

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 4. Load and validate the API keys

`find_dotenv(usecwd=True)` searches from the current working directory. `load_dotenv` loads the values into the process environment, and `os.getenv` reads them.

The validation fails early with a useful message if a key is missing. That is better than allowing a confusing authentication error to appear later during embedding or generation.

In [3]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError(
        "Could not find .env. Put it in the project directory or a parent directory."
    )

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
missing_keys = [
    name
    for name, value in {
        "GEMINI_API_KEY": GEMINI_API_KEY,
        "GROQ_API_KEY": GROQ_API_KEY,
    }.items()
    if not value
]

if missing_keys:
    raise EnvironmentError(
        "Missing required environment variable(s): " + ", ".join(missing_keys)
    )

print("Required API keys are available. Secret values were not displayed.")

Required API keys are available. Secret values were not displayed.


## 5. Configure the embedding and generation providers

We deliberately use separate variables for separate responsibilities:

- **Gemini client + `EMBEDDING_MODEL`:** turns document chunks and search queries into vectors.
- **`llm` (Groq):** generates the final natural-language answer after retrieval.

An embedding model does not generate the answer, and the chat model does not create the vector index in this design. Keeping these roles explicit prevents the original notebook's `llm` variable from being overwritten by a second provider later.

In [4]:
EMBEDDING_MODEL = "gemini-embedding-2"
GENERATION_MODEL = "openai/gpt-oss-20b"  # Model ID served by Groq

client = genai.Client(api_key=GEMINI_API_KEY)
llm = ChatGroq(
    model=GENERATION_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
)

print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)

Embedding model: gemini-embedding-2
Generation model: openai/gpt-oss-20b


## 6. Smoke-test both services

A **smoke test** is a small request used to confirm that the integration can reach the service and receive a response. It is not a quality or performance evaluation.

This cell makes one small embedding request to Gemini and one short generation request to Groq. Skip it when you do not want to use API quota yet; the later embedding and RAG cells also require working credentials.

In [5]:
test_text = "RAG retrieves useful information before generating an answer."
embedding_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=test_text,
)
test_vector = embedding_response.embeddings[0].values

print("Gemini embedding dimensions:", len(test_vector))
print("First five values:", [round(float(x), 5) for x in test_vector[:5]])

llm_response = llm.invoke("Explain RAG in one simple sentence.")
print("\nGroq response:")
print(llm_response.content)

Gemini embedding dimensions: 3072
First five values: [0.01424, 0.0107, 0.00241, 0.00797, 0.01752]

Groq response:
RAG is a technique that blends a search engine with a language model, so the model can pull in relevant documents before generating its answer.


## 7. Create a small knowledge base

For a reproducible exercise, we start with three short documents about machine learning, deep learning, and natural language processing (NLP). In a real application, these documents could come from policies, manuals, reports, or other authorized sources.

The `sample_docs` list holds plain Python strings. We save them to files next so the example also demonstrates a basic file-ingestion workflow.

In [6]:
sample_docs = [
    """Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rule. Three main types are
supervised learning, unsupervised learning, and reinforcement learning. Supervised
learning uses labelled examples; unsupervised learning discovers patterns in unlabelled
data; reinforcement learning learns from actions and rewards in an environment.""",
    """Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language processing, and speech tasks.
Convolutional Neural Networks (CNNs) are commonly used for image-related tasks, while
recurrent networks and Transformers can model sequential or contextual relationships.""",
    """Natural Language Processing (NLP)

Natural Language Processing is the field of AI focused on processing human language.
Tasks include text classification, named entity recognition, sentiment analysis,
translation, summarization, and question answering. Modern NLP commonly uses Transformer
architectures and attention mechanisms to model context and relationships between tokens.""",
]

print(f"Sample documents created: {len(sample_docs)}")

Sample documents created: 3


## 8. Save the sample documents as text files

This cell creates a project-local folder and writes one `.txt` file per sample document. `Path` handles file paths in a platform-independent way; `mkdir(..., exist_ok=True)` makes reruns safe when the folder already exists; `write_text` writes UTF-8 text.

**Why save files?** It separates data from processing code and gives us a realistic input format for the loading step. The cell overwrites only these tutorial sample files with the same names.

In [7]:
DOCS_DIR = Path("data/chromadb_gemini_docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

for index, text in enumerate(sample_docs):
    path = DOCS_DIR / f"doc_{index}.txt"
    path.write_text(text.strip(), encoding="utf-8")
    print("Saved:", path)

print("Total text files:", len(list(DOCS_DIR.glob("doc_*.txt"))))

Saved: data\chromadb_gemini_docs\doc_0.txt
Saved: data\chromadb_gemini_docs\doc_1.txt
Saved: data\chromadb_gemini_docs\doc_2.txt
Total text files: 3


## 9. Load files into LangChain `Document` objects

A LangChain `Document` contains `page_content` plus a `metadata` dictionary. The text is what will be embedded and retrieved; metadata helps identify the source and filter or audit results later.

This cell uses standard-library `pathlib` to read each `.txt` file, avoiding a dependency on a legacy community loader for a simple text-only example. Each document gets:

- `source`: the resolved file path, useful for source attribution.
- `topic`: a stable label derived from the filename, useful for the filtering example later.

In [8]:
documents = []
for path in sorted(DOCS_DIR.glob("doc_*.txt")):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        continue
    documents.append(
        Document(
            page_content=text,
            metadata={"source": str(path.resolve()), "topic": path.stem},
        )
    )

if not documents:
    raise ValueError(
        f"No non-empty .txt documents found in {DOCS_DIR.resolve()}")

print("Loaded documents:", len(documents))
print("First document preview:", documents[0].page_content[:180], "...")
print("First document metadata:", documents[0].metadata)

Loaded documents: 3
First document preview: Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rul ...
First document metadata: {'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_0.txt', 'topic': 'doc_0'}


## 10. Split documents into chunks

A **chunk** is a smaller piece of a larger document. Retrieval usually works better when the indexed units are focused passages, rather than whole long documents.

`RecursiveCharacterTextSplitter` tries paragraph, line, word, and character boundaries in order. The default separators are more robust than using only a space as the separator.

- `chunk_size=500`: target maximum chunk length in characters for this learning example; it is not a token count.
- `chunk_overlap=80`: repeats some text between adjacent chunks to reduce context loss at boundaries.
- `add_start_index=True`: adds the starting character position to metadata for debugging and source inspection.

These values are starting points, not universal optimal settings. Tune them using retrieval tests on your own data.

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)
chunks = text_splitter.split_documents(documents)

if not chunks:
    raise ValueError("Text splitting produced no chunks.")

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print("First chunk preview:", chunks[0].page_content[:180], "...")
print("First chunk metadata:", chunks[0].metadata)

Created 3 chunks from 3 documents
First chunk preview: Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rul ...
First chunk metadata: {'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_0.txt', 'topic': 'doc_0', 'start_index': 0}


## 11. Define a Gemini embedding adapter

An **embedding** is a list of numbers that represents text in a space where semantic similarity can be measured. The model learns this representation; the vector is not a human-readable definition.

LangChain's `Embeddings` interface expects two methods: `embed_documents(texts)` and `embed_query(text)`. The adapter below translates that interface into calls to Google's `google-genai` SDK.

**Important Gemini Embedding 2 detail:** Google recommends explicit task formatting for text-only retrieval. We therefore format documents as `title: none | text: ...` and queries as `task: question answering | query: ...`. Query and document formatting are intentionally different, so both methods must stay consistent with this setup. Do not mix these vectors with another embedding model's vectors; embedding spaces are model-specific.

This simple class sends one request per text. That is acceptable for a tiny learning dataset, but a production ingestion process should consider provider batch APIs, retries/backoff, quota limits, and resumable jobs. Do not pass a list of raw text strings to `gemini-embedding-2` expecting one vector per string: by default, multiple input parts can produce an aggregated embedding.

In [10]:
DOCUMENT_PREFIX = "title: none | text: "
QUERY_PREFIX = "task: question answering | query: "


class GeminiEmbeddings(Embeddings):
    """Adapt Gemini Embedding 2 to LangChain's Embeddings interface."""

    def __init__(self, gemini_client, model: str = EMBEDDING_MODEL):
        self.client = gemini_client
        self.model = model

    def _embed_one(self, text: str) -> list[float]:
        response = self.client.models.embed_content(
            model=self.model,
            contents=text,
        )
        if not response.embeddings or not response.embeddings[0].values:
            raise ValueError("Gemini returned an empty embedding.")
        return [float(value) for value in response.embeddings[0].values]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed_one(DOCUMENT_PREFIX + text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed_one(QUERY_PREFIX + text)


gemini_embeddings = GeminiEmbeddings(client)
print("Gemini embedding adapter is ready.")

Gemini embedding adapter is ready.


## 12. Test the embedding adapter

This is a focused check that the **same adapter** intended for ChromaDB can embed a query. It verifies the vector is non-empty and lets us inspect its dimension.

The number of dimensions matters because every vector stored in one collection must have the same dimensionality. If you change embedding models or output dimensions, rebuild/re-embed the collection rather than mixing incompatible vectors.

In [11]:
sample_vector = gemini_embeddings.embed_query("What is deep learning?")
assert sample_vector and all(isinstance(x, float) for x in sample_vector)

print("Query vector dimensions:", len(sample_vector))
print("First five values:", [round(x, 5) for x in sample_vector[:5]])

Query vector dimensions: 3072
First five values: [-0.01693, 0.01485, 0.01038, 0.02769, -0.00739]


## 13. Cache document embeddings to reduce repeated API calls

**The embedding cache and ChromaDB solve different problems:**

- The JSON cache saves provider-generated vectors so unchanged chunks do not need to be embedded again.
- ChromaDB stores vectors alongside text and metadata and provides similarity retrieval.

The cache is valid only when the model, document formatting prefix, and exact chunk texts match. If any of these change, embeddings must be regenerated. The cache uses JSON for readability in this small exercise; for large corpora, use a more suitable storage format or a managed ingestion pipeline.

In [12]:
chunk_texts = [chunk.page_content for chunk in chunks]
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = ARTIFACTS_DIR / "gemini_chunk_embeddings.json"

cached_data = {}
if CACHE_FILE.exists():
    try:
        cached_data = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        # A missing or invalid cache is recoverable: recompute the vectors.
        cached_data = {}

cache_is_valid = (
    cached_data.get("schema_version") == 1
    and cached_data.get("model") == EMBEDDING_MODEL
    and cached_data.get("document_prefix") == DOCUMENT_PREFIX
    and cached_data.get("texts") == chunk_texts
    and len(cached_data.get("embeddings", [])) == len(chunk_texts)
)

if cache_is_valid:
    chunk_embeddings = [
        [float(value) for value in vector]
        for vector in cached_data["embeddings"]
    ]
    print("Reused cached document embeddings.")
else:
    chunk_embeddings = gemini_embeddings.embed_documents(chunk_texts)
    cache_payload = {
        "schema_version": 1,
        "model": EMBEDDING_MODEL,
        "document_prefix": DOCUMENT_PREFIX,
        "texts": chunk_texts,
        "embeddings": chunk_embeddings,
    }
    CACHE_FILE.write_text(
        json.dumps(cache_payload),
        encoding="utf-8",
    )
    print("Generated and cached document embeddings.")

if len(chunk_embeddings) != len(chunks):
    raise ValueError("Chunk count does not match embedding count.")
if not chunk_embeddings or not chunk_embeddings[0]:
    raise ValueError("No valid document embeddings were generated.")
if len({len(vector) for vector in chunk_embeddings}) != 1:
    raise ValueError("Embedding dimensions are inconsistent.")

print("Chunks:", len(chunks))
print("Embeddings:", len(chunk_embeddings))
print("Vector dimensions:", len(chunk_embeddings[0]))
print("Cache file:", CACHE_FILE.resolve())

Generated and cached document embeddings.
Chunks: 3
Embeddings: 3
Vector dimensions: 3072
Cache file: E:\RAG\Vector_stores_&_Vector_databases\artifacts\gemini_chunk_embeddings.json


## 14. Store precomputed vectors in a persistent ChromaDB collection

ChromaDB is the vector store/database in this local exercise. Each record contains an ID, a vector, the original chunk text, and metadata. `PersistentClient` writes the collection to disk so it can survive a kernel restart.

This cell uses ChromaDB's **public client/collection API** to upsert vectors already produced by the cache step. It then creates a LangChain `Chroma` wrapper around the same collection so query embedding and retrieval fit into LangChain chains.

We use cosine distance for this teaching collection. The clean notebook uses a dedicated directory and collection name; the cell replaces records in that tutorial collection to avoid stale chunks after a rerun. **Do not apply that delete-and-rebuild pattern blindly to a shared or production collection.**

In [13]:
PERSIST_DIRECTORY = ARTIFACTS_DIR / "chroma_db_gemini_groq_clean"
COLLECTION_NAME = "rag_learning_gemini_groq_clean"
PERSIST_DIRECTORY.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(PERSIST_DIRECTORY))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,
    metadata={"hnsw:space": "cosine"},
)

# This is a dedicated tutorial collection, so remove its old sample records first.
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)

chunk_ids = [f"chunk_{index:04d}" for index in range(len(chunks))]
chunk_metadatas = [dict(chunk.metadata) for chunk in chunks]

collection.upsert(
    ids=chunk_ids,
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=chunk_metadatas,
)

# Wrap the same collection with LangChain for retriever/chain integration.
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    embedding_function=gemini_embeddings,
)

print("Collection:", COLLECTION_NAME)
print("Stored records:", collection.count())
print("Persistent directory:", PERSIST_DIRECTORY.resolve())

Collection: rag_learning_gemini_groq_clean
Stored records: 3
Persistent directory: E:\RAG\Vector_stores_&_Vector_databases\artifacts\chroma_db_gemini_groq_clean


## 15. Verify stored text and metadata

Do not trust only a success message. Check that the number of stored records matches the number of chunks and inspect one stored record.

`collection.get` is used here because we want to inspect the database records directly. We do not access LangChain's private `_collection` attribute; private internals can change between package versions.

In [14]:
assert collection.count() == len(chunks), (
    f"Expected {len(chunks)} records, found {collection.count()}"
)

stored = collection.get(
    ids=[chunk_ids[0]],
    include=["documents", "metadatas"],
)
print("Record count:", collection.count())
print("Stored text preview:", stored["documents"][0][:200], "...")
print("Stored metadata:", stored["metadatas"][0])

Record count: 3
Stored text preview: Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rule. Three main types  ...
Stored metadata: {'topic': 'doc_0', 'start_index': 0, 'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_0.txt'}


## 16. Run semantic similarity search

**Semantic search** compares the meaning-oriented query vector with vectors indexed from the document chunks. It can retrieve relevant content even when the wording is not exactly the same, although quality depends on the model, text formatting, chunking, and corpus.

`similarity_search(query, k=3)` returns up to three `Document` objects. Each document includes the matched text and the metadata attached during ingestion.

In [15]:
query = "What are the types of machine learning?"
results = vectorstore.similarity_search(query, k=3)

print("Query:", query)
print(f"Returned {len(results)} result(s):")
for rank, doc in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---")
    print(doc.page_content[:220], "...")
    print("Source:", doc.metadata.get("source", "unknown"))
    print("Topic:", doc.metadata.get("topic", "unknown"))

Query: What are the types of machine learning?
Returned 3 result(s):

--- Result 1 ---
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rule. Three main types are
supervised learn ...
Source: E:\RAG\Vector_stores_&_Vector_databases\data\chromadb_gemini_docs\doc_0.txt
Topic: doc_0

--- Result 2 ---
Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language processing, and speech tasks.
Convolutional  ...
Source: E:\RAG\Vector_stores_&_Vector_databases\data\chromadb_gemini_docs\doc_1.txt
Topic: doc_1

--- Result 3 ---
Natural Language Processing (NLP)

Natural Language Processing is the field of AI focused on processing human language.
Tasks include text classification, named entity recognition, sentiment analysis,
translation, summar ...
Source: 

## 17. Inspect retrieval distances carefully

`similarity_search_with_score` returns a numeric **distance** for ChromaDB, not a universal percentage of relevance. With this collection's cosine-distance configuration, smaller distance means closer vectors; the exact interpretation depends on the selected metric and integration.

Do not say “a score of 0.8 means 80% relevant.” Instead, compare candidates consistently, inspect retrieved text, and set any relevance threshold using an evaluation set for your application.

In [16]:
query = "What is deep learning?"
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

for rank, (doc, distance) in enumerate(results_with_scores, start=1):
    print(f"\n--- Result {rank} | cosine distance: {distance:.4f} ---")
    print(doc.page_content[:180], "...")
    print("Topic:", doc.metadata.get("topic", "unknown"))


--- Result 1 | cosine distance: 0.2557 ---
Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language proc ...
Topic: doc_1

--- Result 2 | cosine distance: 0.3378 ---
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rul ...
Topic: doc_0

--- Result 3 | cosine distance: 0.3770 ---
Natural Language Processing (NLP)

Natural Language Processing is the field of AI focused on processing human language.
Tasks include text classification, named entity recognition, ...
Topic: doc_2


## 18. Filter retrieval by metadata

Metadata filtering is useful when search should be restricted to a subset of the corpus—for example, one document collection, product, tenant, year, or permission scope.

The `filter` below requires the metadata field `topic` to equal `doc_1`. The matching is exact, not substring matching. In production, enforce access-control filters in trusted application code; never rely on the LLM to decide which private records it may see.

In [17]:
filtered_results = vectorstore.similarity_search(
    "What are neural networks used for?",
    k=3,
    filter={"topic": "doc_1"},
)

print("Filtered result count:", len(filtered_results))
for doc in filtered_results:
    print("Topic:", doc.metadata.get("topic"))
    print(doc.page_content[:180], "...")

Filtered result count: 1
Topic: doc_1
Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language proc ...


## 19. Create a retriever

A **vector store** exposes database operations such as similarity search. A **retriever** provides a standard interface that accepts a query and returns relevant documents; LangChain can use it as a component in a larger runnable/chain.

`as_retriever(search_kwargs={"k": 3})` configures how many chunks to request. The retriever does not generate answers—it only retrieves documents.

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("What is NLP?")

print("Retrieved documents:", len(retrieved_docs))
for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved document {rank} ---")
    print(doc.page_content[:180], "...")
    print("Metadata:", doc.metadata)

Retrieved documents: 3

--- Retrieved document 1 ---
Natural Language Processing (NLP)

Natural Language Processing is the field of AI focused on processing human language.
Tasks include text classification, named entity recognition, ...
Metadata: {'topic': 'doc_2', 'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_2.txt', 'start_index': 0}

--- Retrieved document 2 ---
Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language proc ...
Metadata: {'start_index': 0, 'topic': 'doc_1', 'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_1.txt'}

--- Retrieved document 3 ---
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rul ...
Metadata: {'topic': 'doc_0', 'start_index': 0, 'so

## 20. Build a prompt that uses retrieved context

A prompt template separates instructions from runtime inputs. `{context}` will hold the retrieved text; `{input}` will hold the user's question.

The system instruction asks the model to answer from the supplied context and to say when the documents do not support an answer. This reduces unsupported answers but does **not** guarantee perfect grounding, so inspect citations/sources and evaluate the application separately.

In [19]:
system_prompt = """You are a question-answering assistant.
Answer using only the retrieved context below.
If the context does not contain the answer, say that the provided documents do not contain enough information.
Do not invent facts. Be concise.

Retrieved context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Question: {input}"),
])

print("Prompt template created.")

Prompt template created.


## 21. Build the RAG chain with LCEL

LCEL (LangChain Expression Language) composes runnable components using `|` and `.assign(...)`. We use it as the **one executable RAG implementation** in this cleaned notebook, avoiding repeated construction of equivalent chains.

The chain performs these steps:

1. `itemgetter("input") | retriever` sends the question to ChromaDB and stores the retrieved `Document` objects in `retrieved_docs`.
2. `itemgetter("retrieved_docs") | RunnableLambda(format_docs)` joins the document text into a single `context` string.
3. `prompt | llm | StrOutputParser()` formats the question/context, calls Groq, and converts the model result to a plain string.
4. The result keeps `input`, `retrieved_docs`, `context`, and `answer`, so we can inspect both the final answer and evidence without performing retrieval twice.

`RunnablePassthrough.assign(...)` preserves the existing dictionary and adds a new key at each stage. This is useful when debugging RAG because intermediate inputs and retrieved documents remain visible.

In [20]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def format_docs(docs: list[Document]) -> str:
    """Join retrieved document chunks into the prompt context."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    RunnablePassthrough.assign(
        retrieved_docs=itemgetter("input") | retriever,
    )
    .assign(
        context=itemgetter("retrieved_docs") | RunnableLambda(format_docs),
    )
    .assign(
        answer=prompt | llm | StrOutputParser(),
    )
)

print("LCEL RAG chain created.")

LCEL RAG chain created.


## 22. Ask one question and inspect the evidence

The chain accepts a dictionary with the key `input`. It returns an `answer` string and the `retrieved_docs` list, which carries the source metadata from ingestion.

**Debugging rule:** when an answer is wrong, first look at `retrieved_docs`. If the required fact is not there, investigate ingestion, chunking, query/document embedding formatting, metadata filters, or retrieval settings before blaming the language model.

In [21]:
question = "What are the three main types of machine learning?"
response = rag_chain.invoke({"input": question})

print("Question:", response["input"])
print("\nAnswer:")
print(response["answer"])
print("\nRetrieved context sources:")
for rank, doc in enumerate(response["retrieved_docs"], start=1):
    print(f"\n--- Source {rank} ---")
    print(doc.page_content[:220], "...")
    print("Metadata:", doc.metadata)

Question: What are the three main types of machine learning?

Answer:
The three main types of machine learning are:

1. Supervised learning  
2. Unsupervised learning  
3. Reinforcement learning

Retrieved context sources:

--- Source 1 ---
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn
from data rather than being explicitly programmed for every rule. Three main types are
supervised learn ...
Metadata: {'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_0.txt', 'start_index': 0, 'topic': 'doc_0'}

--- Source 2 ---
Deep Learning and Neural Networks

Deep learning is a subset of machine learning based on neural networks with multiple
layers. It is used in computer vision, natural language processing, and speech tasks.
Convolutional  ...
Metadata: {'topic': 'doc_1', 'source': 'E:\\RAG\\Vector_stores_&_Vector_databases\\data\\chromadb_gemini_docs\\doc_1.txt', 'start_index': 0}

--- Sour

## 23. Test several questions

These examples smoke-test a few topic areas in the sample knowledge base. They are not a formal evaluation dataset and do not prove retrieval quality or answer accuracy at scale.

The helper returns the complete chain response, so the caller can later inspect or save the context and answer rather than only printing a string.

In [22]:
def ask_rag(question: str) -> dict:
    """Run the RAG chain and print the answer with retrieved-source metadata."""
    result = rag_chain.invoke({"input": question})
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Retrieved topics:", [doc.metadata.get("topic")
          for doc in result["retrieved_docs"]])
    return result


test_questions = [
    "What are the three types of machine learning?",
    "How does deep learning relate to neural networks?",
    "Which tasks are included in NLP?",
]

responses = [ask_rag(question) for question in test_questions]

Question: What are the three types of machine learning?
Answer: The three main types of machine learning are:

1. Supervised learning  
2. Unsupervised learning  
3. Reinforcement learning
Retrieved topics: ['doc_0', 'doc_1', 'doc_2']
Question: How does deep learning relate to neural networks?
Answer: Deep learning is a subset of machine learning that specifically uses neural networks with multiple layers. It builds on the concept of neural networks by adding depth—additional hidden layers—to capture more complex patterns.
Retrieved topics: ['doc_1', 'doc_0', 'doc_2']
Question: Which tasks are included in NLP?
Answer: The tasks included in NLP are:

- Text classification  
- Named entity recognition  
- Sentiment analysis  
- Translation  
- Summarization  
- Question answering
Retrieved topics: ['doc_2', 'doc_1', 'doc_0']


## 24. How this maps to older LangChain chain helpers

Older tutorials often use two high-level functions:

- **`create_stuff_documents_chain`** combines retrieved `Document` objects into the `{context}` prompt variable and calls the language model. “Stuff” means the selected documents are placed together into one context; this is simple but can overwhelm the model with long or irrelevant passages.
- **`create_retrieval_chain`** connects a retriever to a document-combination chain. It retrieves documents, passes them downstream, and returns the answer along with context.

Conceptually, the LCEL pipeline above performs the same jobs, but the individual data-flow steps are explicit and it preserves the retrieved `Document` objects for inspection. You may encounter the helper functions in codebases and interviews, so understand their purpose even if you build new projects with LCEL.

**Version note (reviewed 26 October 2026):** the [official `langchain-classic` reference](https://reference.langchain.com/python/langchain-classic/chains) says that package is maintained for security vulnerabilities until December 2026 and encourages migration to current LangChain. This cleaned notebook therefore does not require it.

## 25. Interview-ready recap and troubleshooting

### Explain the architecture in 30 seconds

“I load source documents, split them into chunks, and generate document embeddings. I persist each vector with its text and metadata in ChromaDB. At query time, the same embedding adapter embeds the question, Chroma retrieves the nearest chunks, and a retriever passes them into a prompt. Groq generates an answer from that context, and I inspect the answer alongside the retrieved documents.”

### Definitions to know

- **RAG:** a pattern that retrieves external evidence and supplies it to a generative model at answer time.
- **Embedding:** a numerical vector representation produced by an embedding model.
- **Chunking:** splitting source content into retrievable units.
- **Vector store:** an abstraction for storing vectors and associated payloads and running vector similarity operations.
- **Vector database:** a database system designed to manage vector records and retrieval, often with persistence, filtering, concurrency, operational tooling, and scaling features.
- **Similarity search:** ranking vectors by a configured distance/similarity metric.
- **Retriever:** a component that accepts a query and returns relevant documents.
- **Metadata:** structured fields stored alongside a document, such as source path, topic, or timestamps.
- **Embedding cache:** a separate cache that avoids recomputing the same provider embeddings; it is not the retrieval index.
- **LCEL:** LangChain's compositional runnable syntax for connecting processing steps.

### Common debugging sequence

1. **Authentication error:** verify `.env` is found and both keys exist without printing them.
2. **Embedding API error:** verify the model ID, quota, network, and request format.
3. **Dimension mismatch:** never mix models or embedding dimensions in one collection; re-embed all records after changing the model/configuration.
4. **Poor retrieval:** inspect chunk size/overlap, query/document prefixes, retrieved text, metadata, and the distance metric before changing the generator.
5. **Answer not grounded:** verify that the right context was retrieved; tighten the prompt, reduce irrelevant context, and evaluate answers.
6. **Repeated API calls:** confirm that cache model, document formatting, and chunk texts match. The cache should be invalidated whenever embedding inputs change.
7. **Stale database records:** use deterministic IDs and a deliberate update/delete policy. The rebuild logic in this notebook is only for its dedicated learning collection.

### Important limitations

This is a small local learning project. It is not production-hardened: it does not include a formal evaluation set, retries/backoff, batching at scale, access-control integration, automated source citations, monitoring, or deployment. Do not claim those capabilities based on this notebook alone.

## Official references

- [Google Gemini embeddings documentation](https://ai.google.dev/gemini-api/docs/embeddings) — model IDs, embedding usage, text task formatting, and batching notes.
- [LangChain Chroma integration reference](https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma) — current `langchain-chroma` integration and public vector-store API.
- [Groq GPT-OSS 20B model documentation](https://console.groq.com/docs/model/openai/gpt-oss-20b) — generation model ID used in the notebook.

**Final practice task:** close the notes, explain the pipeline out loud, and then rebuild it from an empty notebook. You understand it when you can explain why every component exists, inspect the retrieved context, and debug a failed retrieval without guessing.